In [1]:
import sys
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start, *start.parents]
    for p in candidates:
        if (p / 'module').exists() and (p / 'configs').exists():
            return p
    fallback = Path('/N/u/kmluong/BigRed200/regDL-TCIP')
    if fallback.exists():
        return fallback
    raise FileNotFoundError('Cannot find repo root (folder containing module/ and configs/).')


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from visualization.model_loader import load_model_from_paths
from module.training.datasets import load_split_arrays
from module.training.masks import extract_bc_rim_from_y, make_rim_mask_like, make_smooth_phi


In [2]:
def load_data_array(data_source: str | Path, split: str = 'test'):
    p = Path(data_source)
    if p.is_dir():
        train, val, test = load_split_arrays(p)
        split = split.lower()
        if split == 'train':
            arr = train
        elif split == 'val':
            if val is None:
                raise ValueError('No val.npy found in this temp dir.')
            arr = val
        elif split == 'test':
            arr = test
        else:
            raise ValueError(f'Invalid split: {split}')
        source_kind = f'temp_dir:{split}'
    else:
        if p.suffix != '.npy':
            raise ValueError(f'DATA_SOURCE must be temp dir or .npy file, got: {p}')
        arr = np.load(p, mmap_mode='r')
        source_kind = 'npy'

    if arr.ndim != 5:
        raise ValueError(f'Data must have shape [N,F,H,W,C], got: {arr.shape}')

    return arr, source_kind


def make_window_tensors(arr, n_idx: int, f_idx: int, step_in: int, device: str):
    N, F, H, W, C = arr.shape
    if not (0 <= n_idx < N):
        raise IndexError(f'SAMPLE_N out of range [0, {N-1}]')
    if not (0 <= f_idx <= F - step_in - 1):
        raise IndexError(
            f'SAMPLE_F out of range [0, {F-step_in-1}] to keep a valid target frame (F={F}, step_in={step_in})'
        )

    x_np = arr[n_idx, f_idx : f_idx + step_in, ...]
    y_np = arr[n_idx, f_idx + step_in, ...]

    # [T,H,W,C] -> [1,T,C,H,W]
    x = torch.from_numpy(np.transpose(x_np, (0, 3, 1, 2)).copy()).float().unsqueeze(0).to(device)
    # [H,W,C] -> [1,C,H,W]
    y = torch.from_numpy(np.transpose(y_np, (2, 0, 1)).copy()).float().unsqueeze(0).to(device)
    return x, y


In [ ]:
# ===== 12-panel CMIP6 figure + storm-centered coordinate maps from Z =====
# Shape assumption:
#   arr: (sample, frame, h, w, c)
#   Z:   (sample, frame, >=4) = [lon, lat, sinDOY, cosDOY, ...]
#
# In maps  = first 3 frames
# Out map  = last / target frame
# Plot     = frame 0 and channel 0 equivalent from model window

from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

from visualization.latlon import (
    extract_centers_from_z,
    extract_latlon_crops_for_track,
    get_extent_from_latlon,
)

# =========================
# User inputs
# =========================
DS = "CMIP6"
DATA_SOURCE = f"/N/slate/kmluong/regDL-TCIP/{DS}/tmp"
LEVEL2_SOURCE = f"/N/slate/kmluong/regDL-TCIP/{DS}/level_2_data"

SPLIT = "test"
SAMPLE_N = 150
SAMPLE_F = 0
CHANNEL_TO_SHOW = 0
RIM = 1
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CM6_LAT_PATH = f"/N/u/kmluong/BigRed200/Deep-learning-intensity-projection/test_space/cm6lat.npy"
CM6_LON_PATH = f"/N/u/kmluong/BigRed200/Deep-learning-intensity-projection/test_space/cm6lon.npy"

# Try these Z paths in order
Z_CANDIDATES = [
    f"{DATA_SOURCE}/{SPLIT}_Z.npy",
    f"{DATA_SOURCE}/Z.npy",
    f"{LEVEL2_SOURCE}/wrf_tropical_cyclone_track_5f_12v_dataset_Z.npy",
]

MODEL_CONFIG_PATH_BC = f"/N/slate/kmluong/regDL-TCIP/{DS}/checkpoints/AFNO-TCP-BC.json"
MODEL_CHECKPOINT_PATH_BC = f"/N/slate/kmluong/regDL-TCIP/{DS}/checkpoints/trained-AFNO-TCP-BC.pt"

MODEL_CONFIG_PATH_NBC = f"/N/slate/kmluong/regDL-TCIP/{DS}/checkpoints/AFNO-TCP-NO-BC.json"
MODEL_CHECKPOINT_PATH_NBC = f"/N/slate/kmluong/regDL-TCIP/{DS}/checkpoints/trained-AFNO-TCP-NO-BC.pt"


# =========================
# Helpers assumed available:
# load_model_from_paths, load_data_array, extract_bc_rim_from_y,
# make_rim_mask_like, make_smooth_phi, make_window_tensors
# =========================
def _run_inference_one_window(model, x, y_true, use_bc: bool, rim: int = 1):
    with torch.no_grad():
        x_in = x
        y_in = y_true

        if getattr(model, "x_scaler", None) is not None:
            x_in = model.x_scaler.norm(x_in)
        if getattr(model, "y_scaler", None) is not None:
            y_in = model.y_scaler.norm(y_in)

        if use_bc:
            b_fill = extract_bc_rim_from_y(y_in, rim=rim)
            bc_mask = make_rim_mask_like(y_in, rim=rim)
            bc_in = torch.cat([b_fill, bc_mask], dim=1)

            y_free = model(x_in, bc_in)

            phi = make_smooth_phi(
                H=y_in.shape[-2],
                W=y_in.shape[-1],
                rim=rim,
                device=y_in.device,
                dtype=y_in.dtype,
            )
            y_pred_norm = phi * y_free + (1.0 - phi) * b_fill
        else:
            y_pred_norm = model(x_in)

        if getattr(model, "y_scaler", None) is not None:
            y_pred = model.y_scaler.denorm(y_pred_norm)
            y_ref = y_true
        else:
            y_pred = y_pred_norm
            y_ref = y_true

    return y_pred, y_ref


def find_existing_path(paths):
    """Return first existing path from candidate list."""
    for p in paths:
        if Path(p).exists():
            return p
    raise FileNotFoundError("No Z file found. Tried:\n" + "\n".join(paths))


# =========================
# Load data
# =========================
arr, _ = load_data_array(DATA_SOURCE, split=SPLIT)
print("arr shape:", arr.shape)

z_path = find_existing_path(Z_CANDIDATES)
Z = np.load(z_path, mmap_mode="r")
print("Z path:", z_path)
print("Z shape:", Z.shape)

cm6lat = np.load(CM6_LAT_PATH)[0]
cm6lon = np.load(CM6_LON_PATH)[0]
print("cm6lat shape:", cm6lat.shape)
print("cm6lon shape:", cm6lon.shape)


# =========================
# Load models
# =========================
model_bc, meta_bc = load_model_from_paths(
    config_path=MODEL_CONFIG_PATH_BC,
    checkpoint_path=MODEL_CHECKPOINT_PATH_BC,
    map_location=DEVICE,
    strict=True,
    eval_mode=True,
)
model_bc = model_bc.to(DEVICE)
step_in_bc = int(meta_bc["config"]["num_times"])

model_nbc, meta_nbc = load_model_from_paths(
    config_path=MODEL_CONFIG_PATH_NBC,
    checkpoint_path=MODEL_CHECKPOINT_PATH_NBC,
    map_location=DEVICE,
    strict=True,
    eval_mode=True,
)
model_nbc = model_nbc.to(DEVICE)
step_in_nbc = int(meta_nbc["config"]["num_times"])

step_in = min(step_in_bc, step_in_nbc)
print("step_in:", step_in)


# =========================
# Build model window
# =========================
x, y_true = make_window_tensors(arr, SAMPLE_N, SAMPLE_F, step_in=step_in, device=DEVICE)

T = x.shape[1]
if T < 3:
    raise ValueError("Need >=3 input frames")

c = int(CHANNEL_TO_SHOW)

in_maps = [x[0, t, c].detach().cpu().numpy() for t in range(3)]
truth_map = y_true[0, c].detach().cpu().numpy()


# =========================
# Extract lon/lat from Z
# =========================
# Current step2 convention: [lon, lat, sinDOY, cosDOY, ...]
frame_ids = [
    SAMPLE_F,
    SAMPLE_F + 1,
    SAMPLE_F + 2,
    SAMPLE_F + step_in,
]

lo, la = extract_centers_from_z(
    z=Z,
    sample_idx=SAMPLE_N,
    frame_indices=frame_ids,
    feature_order="lon_lat",
)

print("
frame_ids:", frame_ids)
print("lo:", lo)
print("la:", la)


# =========================
# Extract 100x100 lat/lon maps
# =========================
crop_result = extract_latlon_crops_for_track(
    lat2d=cm6lat,
    lon2d=cm6lon,
    lon_centers=lo,
    lat_centers=la,
    size=100,
)
lat_crops = crop_result.lat_crops
lon_crops = crop_result.lon_crops
crop_indices = crop_result.crop_indices
center_indices = crop_result.center_indices

for k, (lo_k, la_k, crop_idx, center_idx) in enumerate(zip(lo, la, crop_indices, center_indices)):
    iy, ix = center_idx
    print(f"
Map point {k}")
    print("target lon/lat:", lo_k, la_k)
    print("nearest iy/ix:", iy, ix)
    print("nearest lon/lat:", cm6lon[iy, ix], cm6lat[iy, ix])
    print("crop y0/y1/x0/x1:", crop_idx)
    print("center position inside crop:", iy - crop_idx[0], ix - crop_idx[2])

in_extents = [
    get_extent_from_latlon(lat_crops[0], lon_crops[0]),
    get_extent_from_latlon(lat_crops[1], lon_crops[1]),
    get_extent_from_latlon(lat_crops[2], lon_crops[2]),
]
out_extent = get_extent_from_latlon(lat_crops[3], lon_crops[3])

print("
Input extents:", in_extents)
print("Output extent:", out_extent)


# =========================
# Predict
# =========================
y_pred_bc, _ = _run_inference_one_window(model_bc, x, y_true, use_bc=True, rim=RIM)
y_pred_nbc, _ = _run_inference_one_window(model_nbc, x, y_true, use_bc=False, rim=RIM)

pred_bc_map = y_pred_bc[0, c].detach().cpu().numpy()
pred_nbc_map = y_pred_nbc[0, c].detach().cpu().numpy()


# =========================
# Diagnostics
# =========================
d_truth_bc = truth_map - pred_bc_map
d_truth_nbc = truth_map - pred_nbc_map
d_bc_nbc = pred_bc_map - pred_nbc_map


# =========================
# Color scales
# =========================
field_maps = in_maps + [pred_bc_map, pred_nbc_map, truth_map]
vmin = min(float(np.nanmin(m)) for m in field_maps)
vmax = max(float(np.nanmax(m)) for m in field_maps)

diff_maps = [d_truth_bc, d_truth_nbc, d_bc_nbc]
dmax = max(float(np.nanmax(np.abs(m))) for m in diff_maps)
d_vmin, d_vmax = -dmax, dmax


# =========================
# Plot helpers
# =========================
def _add_track_arrows(ax, lon_points, lat_points, color="white"):
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    ax.plot(
        lon_points,
        lat_points,
        marker="o",
        markersize=3.5,
        linewidth=1.0,
        color=color,
        markeredgecolor="black",
        markeredgewidth=0.4,
        alpha=0.95,
        zorder=5,
    )
    for lon0, lat0, lon1, lat1 in zip(lon_points[:-1], lat_points[:-1], lon_points[1:], lat_points[1:]):
        ax.annotate(
            "",
            xy=(lon1, lat1),
            xytext=(lon0, lat0),
            arrowprops={"arrowstyle": "->", "color": color, "lw": 1.4, "shrinkA": 2, "shrinkB": 2},
            zorder=6,
        )
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)


# =========================
# Plot
# =========================
fig, axes = plt.subplots(3, 3, figsize=(14, 14), sharex=False, sharey=False)

# Row 1: Inputs
for j in range(3):
    im_field = axes[0, j].imshow(
        in_maps[j],
        origin="lower",
        extent=in_extents[j],
        vmin=vmin,
        vmax=vmax,
        cmap="viridis",
        aspect="auto",
    )
    _add_track_arrows(axes[0, j], lo[: j + 2], la[: j + 2])

# Row 2: Predictions + Truth
axes[1, 0].imshow(pred_nbc_map, origin="lower", extent=out_extent,
                  vmin=vmin, vmax=vmax, cmap="viridis", aspect="auto")

axes[1, 1].imshow(pred_bc_map, origin="lower", extent=out_extent,
                  vmin=vmin, vmax=vmax, cmap="viridis", aspect="auto")

axes[1, 2].imshow(truth_map, origin="lower", extent=out_extent,
                  vmin=vmin, vmax=vmax, cmap="viridis", aspect="auto")

for ax in axes[1, :]:
    _add_track_arrows(ax, lo, la)

# Row 3: Diagnostics
im_diff = axes[2, 0].imshow(d_truth_nbc, origin="lower", extent=out_extent,
                           vmin=d_vmin, vmax=d_vmax, cmap="coolwarm", aspect="auto")

axes[2, 1].imshow(d_truth_bc, origin="lower", extent=out_extent,
                  vmin=d_vmin, vmax=d_vmax, cmap="coolwarm", aspect="auto")

axes[2, 2].imshow(d_bc_nbc, origin="lower", extent=out_extent,
                  vmin=d_vmin, vmax=d_vmax, cmap="coolwarm", aspect="auto")

for ax in axes[2, :]:
    _add_track_arrows(ax, lo, la, color="black")

# =========================
# Titles
# =========================
titles = [
    ["Input frame 1", "Input frame 2", "Input frame 3"],
    ["Pred NO-BC", "Pred BC", "Truth"],
    ["Truth-NOBC", "Truth-BC", "BC-NOBC"],
]

for i in range(3):
    for j in range(3):
        axes[i, j].set_title(titles[i][j])
        axes[i, j].set_xlabel("Longitude")
        axes[i, j].set_ylabel("Latitude")
        axes[i, j].tick_params(labelsize=8)

# =========================
# TOP COLORBAR 
# =========================
cbar_ax_top = fig.add_axes([0.15, 0.92, 0.7, 0.02])  
# [left, bottom, width, height] (figure coords)

cbar_top = fig.colorbar(
    im_field,
    cax=cbar_ax_top,
    orientation="horizontal"
)
cbar_top.set_label("10m U wind speed (m/s)")
cbar_top.ax.xaxis.set_label_position('top')


# =========================
# BOTTOM COLORBAR
# =========================
cbar_ax_bottom = fig.add_axes([0.15, 0.07, 0.7, 0.02])

cbar_bottom = fig.colorbar(
    im_diff,
    cax=cbar_ax_bottom,
    orientation="horizontal"
)
cbar_bottom.set_label("Difference")


plt.savefig("visualization_cmip6_map.png", dpi=200, bbox_inches="tight")
plt.show()